<a href="https://colab.research.google.com/github/astr-droid/manifold-based-music-recommendations/blob/main/ManifoldLearningForMusic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actual

In [ ]:
!git clone https://github.com/astr-droid/manifold-based-music-recommendations.git

In [ ]:
import pandas as pd

# Load your Spotify track features CSV
df = pd.read_csv("/content/manifold-based-music-recommendations/SpotifyAudioFeaturesApril2019.csv")

# Preview
print(df.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Choose audio features
features = ['danceability', 'energy', 'acousticness', 'instrumentalness', 'tempo', 'valence']

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(df[features])


In [ ]:
import umap.umap_ as umap  # alias to avoid naming issues

reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, metric='euclidean', random_state=42)
embedding = reducer.fit_transform(X)


In [ ]:
from sklearn.neighbors import NearestNeighbors

# Create model from embedded coordinates
nn_model = NearestNeighbors(n_neighbors=6)
nn_model.fit(embedding)


In [ ]:
def recommend(track_id, top_k=5):
    # Filter for a unique track
    match = df[df.track_id.values == track_id]
    if match.empty:
        return f"No match found for {track_id}"

    idx = match.index[0]
    distances, indices = nn_model.kneighbors([embedding[idx]], n_neighbors=top_k + 1)

    # Get recommended track info (excluding the track itself)
    rec_indices = indices[0][1:]
    return df.iloc[rec_indices][['track_name', 'artist_name']]

In [ ]:
print("Recommended tracks:")
print(recommend("22nebdx3j16qQKRrm0mVe7")) # sick boy by the chainsmokers

# Run after presentation:

In [ ]:
!rm -rf manifold-based-music-recommendations